# 02 — DCGAN: 280K Game Sprite Dataset

**Project:** Prompt-to-Game-Asset Generator  
**Model:** Deep Convolutional GAN (DCGAN)  
**Role:** Main convolutional unconditional sprite-generation candidate, compared against the Vanilla GAN baseline.

This notebook intentionally follows `01_Vanilla_GAN_280K_Training.ipynb` for shared dataset, preprocessing, resolution, RGBA handling, normalization, latent dimension, optimizer settings, split proportions, global batch-size convention, checkpoint frequency, fixed samples, evaluation count, and artifact workflow. The architecture is the key planned difference: DCGAN uses convolutional layers.

**Do not edit or restart the active Vanilla GAN run.** This notebook writes only to `/kaggle/working/dcgan_280k/` and creates `DCGAN_280K_COMPLETE.zip`.

The notebook starts with a 50-epoch total target. Set `TARGET_EPOCHS = 100` to continue from a valid checkpoint, not to add 100 more epochs. Kaggle working storage is temporary; download or save a versioned output archive before the session disappears.

---
## Section 2: Environment and GPU Setup

In [ ]:
import os, sys, json, csv, time, random, hashlib, shutil, zipfile, platform, io
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
from pathlib import Path

print(f"Python      : {sys.version}")
print(f"TensorFlow  : {tf.__version__}")
print(f"Platform    : {platform.platform()}")
print(f"Timestamp   : {datetime.utcnow().isoformat()}Z")

# ── GPU detection ────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
print(f"\nGPUs found  : {len(gpus)}")
for g in gpus:
    print(f"  {g.name}")
    try:
        tf.config.experimental.set_memory_growth(g, True)
        print(f"    memory growth: enabled")
    except Exception as e:
        print(f"    memory growth: {e}")

# ── Distribution strategy ────────────────────────────────────────────────────
if len(gpus) >= 2:
    strategy = tf.distribute.MirroredStrategy()
    print(f"\nStrategy    : MirroredStrategy ({strategy.num_replicas_in_sync} replicas)")
elif len(gpus) == 1:
    strategy = tf.distribute.OneDeviceStrategy('/GPU:0')
    print("\nStrategy    : OneDeviceStrategy (1 GPU)")
else:
    strategy = tf.distribute.OneDeviceStrategy('/CPU:0')
    print("\nWARNING: No GPU detected. CPU only — do NOT run full training on CPU.")
    print("Switch Kaggle accelerator to GPU T4 x2 before running the training cell.")

NUM_REPLICAS = strategy.num_replicas_in_sync

# ── Seeds ────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print(f"\nSeed        : {SEED} (locked)")

---
## Section 3: Central Configuration

**All key settings are here. Change `TARGET_EPOCHS` to extend training (e.g. 50 → 100) and re-run.**

Settings are kept identical to Vanilla GAN (notebook 01) wherever possible for fair comparison.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  CENTRAL CONFIGURATION — edit here only
# ═══════════════════════════════════════════════════════════════════════════════

# ── Dataset (identical to Vanilla GAN for fair comparison) ───────────────────
HF_DATASET_NAME        = "evilsocket/alucard-sprites"
IMG_HEIGHT             = 64       # same as Vanilla GAN
IMG_WIDTH              = 64
CHANNELS               = 4        # RGBA — preserve alpha channel
PIXEL_NORM             = "tanh"   # normalize to [-1,1]; generator output = tanh
TRAIN_RATIO            = 0.90
VAL_RATIO              = 0.05

# ── Architecture (DCGAN-specific) ────────────────────────────────────────────
LATENT_DIM             = 100      # same as Vanilla GAN
# Generator channel widths: latent → 4x4x512 → 8x8x256 → 16x16x128 → 32x32x64 → 64x64x4
G_CHANNELS             = [512, 256, 128, 64]   # per upsample block
# Discriminator channel widths: 64x64x4 → 32x32x64 → 16x16x128 → 8x8x256 → 4x4x512 → logit
D_CHANNELS             = [64, 128, 256, 512]   # per downsample block
G_INIT_SPATIAL         = 4        # initial spatial size before upsampling
USE_WEIGHT_INIT        = True     # DCGAN-style normal init (mean=0, std=0.02)

# ── Optimization (identical to Vanilla GAN) ──────────────────────────────────
G_LR                   = 2e-4
D_LR                   = 2e-4
ADAM_BETA1             = 0.5
ADAM_BETA2             = 0.999

# ── Training ─────────────────────────────────────────────────────────────────
TARGET_EPOCHS          = 50       # <-- change to 100, 150, etc. to extend
BATCH_SIZE_PER_REPLICA = 64       # same as Vanilla GAN
GLOBAL_BATCH_SIZE      = BATCH_SIZE_PER_REPLICA * NUM_REPLICAS
RESUME_MODE            = "auto"   # "auto" | "fresh" | "required"

# ── Mixed precision ──────────────────────────────────────────────────────────
USE_MIXED_PRECISION    = False    # set True only if stable on your hardware
if USE_MIXED_PRECISION:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print("Mixed precision: float16 enabled")

# ── Intervals ────────────────────────────────────────────────────────────────
CHECKPOINT_EVERY       = 1        # save resumable checkpoint every N epochs
SAMPLE_EVERY           = 5        # generate sample grid every N epochs
N_FIXED_SAMPLES        = 25       # fixed latent vectors for longitudinal comparison (5x5)
N_EVAL_SAMPLES         = 100      # samples for diversity evaluation
LOG_EVERY              = 1

# Optional: set to a mounted ZIP from a previous DCGAN run to recover in a fresh session.
# Example: /kaggle/input/my-dcgan-output/DCGAN_280K_COMPLETE.zip
# Leave empty for a normal run or when checkpoints already exist in BASE_DIR.
RESUME_ARCHIVE_PATH    = ""

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR               = "/kaggle/working/dcgan_280k"
CKPT_DIR               = f"{BASE_DIR}/checkpoints"
LOG_DIR                = f"{BASE_DIR}/logs"
SAMPLE_DIR             = f"{BASE_DIR}/samples"
CONFIG_DIR             = f"{BASE_DIR}/config"
WEIGHTS_DIR            = f"{BASE_DIR}/weights"
EXPORT_DIR             = f"{BASE_DIR}/export"
ZIP_PATH               = "/kaggle/working/DCGAN_280K_COMPLETE.zip"

# ── Create directories ───────────────────────────────────────────────────────
for d in [CKPT_DIR, LOG_DIR, SAMPLE_DIR, CONFIG_DIR, WEIGHTS_DIR, EXPORT_DIR]:
    os.makedirs(d, exist_ok=True)

# Restore an exported DCGAN archive BEFORE dataset split/model initialization, so the saved
# split manifest and configuration are available to later cells. Never extract over a local checkpoint.
if RESUME_ARCHIVE_PATH:
    if not os.path.isfile(RESUME_ARCHIVE_PATH):
        raise FileNotFoundError(f"RESUME_ARCHIVE_PATH does not exist: {RESUME_ARCHIVE_PATH}")
    if tf.train.latest_checkpoint(CKPT_DIR):
        print("Local DCGAN checkpoint exists; preserving it and not extracting the recovery archive.")
    else:
        with zipfile.ZipFile(RESUME_ARCHIVE_PATH, "r") as _zf:
            _bad = _zf.testzip()
            if _bad is not None:
                raise RuntimeError(f"Recovery ZIP is corrupt at member: {_bad}")
            _root = os.path.abspath(BASE_DIR)
            for _member in _zf.namelist():
                _target = os.path.abspath(os.path.join(BASE_DIR, _member))
                if _target != _root and not _target.startswith(_root + os.sep):
                    raise RuntimeError(f"Unsafe path in recovery archive: {_member}")
            _zf.extractall(BASE_DIR)
        print("Recovery archive extracted before dataset discovery and split creation.")

# ── Validate config ──────────────────────────────────────────────────────────
assert CHANNELS in (3, 4), "CHANNELS must be 3 or 4"
assert PIXEL_NORM in ("tanh", "sigmoid")
assert LATENT_DIM > 0
assert TARGET_EPOCHS > 0
assert RESUME_MODE in ("auto", "fresh", "required")
# Verify generator spatial progression: G_INIT_SPATIAL * 2^len(G_CHANNELS) == IMG_HEIGHT
assert G_INIT_SPATIAL * (2 ** len(G_CHANNELS)) == IMG_HEIGHT, \
    f"G_CHANNELS depth {len(G_CHANNELS)} with init {G_INIT_SPATIAL} does not reach {IMG_HEIGHT}"

CFG = dict(
    model="DCGAN", hf_dataset=HF_DATASET_NAME,
    img_height=IMG_HEIGHT, img_width=IMG_WIDTH, channels=CHANNELS,
    pixel_norm=PIXEL_NORM, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO,
    latent_dim=LATENT_DIM, g_channels=G_CHANNELS, d_channels=D_CHANNELS,
    g_init_spatial=G_INIT_SPATIAL,
    g_lr=G_LR, d_lr=D_LR, beta1=ADAM_BETA1, beta2=ADAM_BETA2,
    target_epochs=TARGET_EPOCHS, batch_size_per_replica=BATCH_SIZE_PER_REPLICA,
    global_batch_size=GLOBAL_BATCH_SIZE, seed=SEED,
    checkpoint_every=CHECKPOINT_EVERY, sample_every=SAMPLE_EVERY, log_every=LOG_EVERY,
    n_fixed_samples=N_FIXED_SAMPLES, n_eval_samples=N_EVAL_SAMPLES, resume_mode=RESUME_MODE,
    resume_archive_path=RESUME_ARCHIVE_PATH,
    base_dir=BASE_DIR, created_at=datetime.utcnow().isoformat()
)
# The authoritative run_config.json is written in the checkpoint/resume section,
# after any recovery archive has been extracted, so a previous config is not overwritten prematurely.
print("Configuration prepared; persistence occurs after resume metadata is inspected.")
print(f"  Model       : DCGAN (convolutional)")
print(f"  Resolution  : {IMG_HEIGHT}x{IMG_WIDTH}x{CHANNELS}")
print(f"  Latent dim  : {LATENT_DIM}")
print(f"  Batch size  : {GLOBAL_BATCH_SIZE} (global)")
print(f"  Target epochs: {TARGET_EPOCHS}")
print(f"  Resume mode : {RESUME_MODE}")
print(f"  G progression: z({LATENT_DIM}) -> {G_INIT_SPATIAL}x{G_INIT_SPATIAL}x{G_CHANNELS[0]} -> ... -> {IMG_HEIGHT}x{IMG_WIDTH}x{CHANNELS}")

---
## Section 4: Dataset Discovery

In [ ]:
# ── Install datasets library if needed ───────────────────────────────────────
try:
    from datasets import load_dataset as hf_load
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)
    from datasets import load_dataset as hf_load

print(f"Loading {HF_DATASET_NAME} ...")
raw_dataset = hf_load(HF_DATASET_NAME)
raw_train   = raw_dataset["train"]
print(f"Raw dataset size: {len(raw_train):,} images")

# ── Inspect a sample image ───────────────────────────────────────────────────
sample_item = raw_train[0]
sample_img  = sample_item["image"]
print(f"Sample image mode : {sample_img.mode}")
print(f"Sample image size : {sample_img.size}  (W x H)")
sample_arr = np.array(sample_img)
print(f"Sample array shape: {sample_arr.shape}")
print(f"Sample dtype      : {sample_arr.dtype}")
print(f"Sample value range: [{sample_arr.min()}, {sample_arr.max()}]")

# ── Deduplication (load cached indices if available) ─────────────────────────
UNIQUE_IDX_PATH   = f"{CONFIG_DIR}/unique_indices.json"
SHUFFLED_IDX_PATH = f"{CONFIG_DIR}/shuffled_indices.json"
SPLIT_META_PATH   = f"{CONFIG_DIR}/split_metadata.json"

if os.path.exists(UNIQUE_IDX_PATH):
    print("\nLoading cached unique indices ...")
    with open(UNIQUE_IDX_PATH) as f:
        unique_indices = json.load(f)
    print(f"Unique images (cached): {len(unique_indices):,}")
else:
    print("\nDeduplicating by SHA-256 pixel hash ...")
    seen, unique_indices = set(), []
    rejected = 0
    for i, item in enumerate(raw_train):
        if i % 10000 == 0:
            print(f"  {i:,}/{len(raw_train):,}  unique so far: {len(unique_indices):,}")
        try:
            arr = np.asarray(item["image"].convert("RGBA"), dtype=np.uint8)
            if arr.shape != (128, 128, 4):
                rejected += 1; continue
            h = hashlib.sha256(arr.tobytes()).hexdigest()
            if h not in seen:
                seen.add(h)
                unique_indices.append(i)
        except Exception:
            rejected += 1
    with open(UNIQUE_IDX_PATH, "w") as f:
        json.dump(unique_indices, f)
    print(f"Deduplication complete: {len(unique_indices):,} unique, {rejected} rejected")

unique_dataset = raw_train.select(unique_indices)
print(f"\nDataset ready: {len(unique_dataset):,} unique sprites")

---
## Section 5: Preprocessing and tf.data Pipeline

In [ ]:
# ── Normalization helpers ─────────────────────────────────────────────────────
def normalize(arr_float32):
    """[0,1] float32 -> [-1,1] for tanh."""
    if PIXEL_NORM == "tanh":
        return arr_float32 * 2.0 - 1.0
    return arr_float32

def denormalize(arr):
    """Invert normalize() for display."""
    if PIXEL_NORM == "tanh":
        return np.clip((arr + 1.0) / 2.0, 0.0, 1.0)
    return np.clip(arr, 0.0, 1.0)

def preprocess_image(item):
    """PIL image -> normalized float32 numpy array at training resolution."""
    from PIL import Image as PILImage
    img = item["image"].convert("RGBA")
    if img.size != (IMG_WIDTH, IMG_HEIGHT):
        img = img.resize((IMG_WIDTH, IMG_HEIGHT), PILImage.LANCZOS)
    arr = np.array(img, dtype=np.float32) / 255.0
    return normalize(arr)

def make_generator_fn(hf_ds):
    def _gen():
        for item in hf_ds:
            yield preprocess_image(item)
    return _gen

def build_tf_dataset(hf_ds, batch_size, shuffle=True, repeat=True):
    ds = tf.data.Dataset.from_generator(
        make_generator_fn(hf_ds),
        output_signature=tf.TensorSpec(shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS), dtype=tf.float32)
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=min(5000, len(hf_ds)), seed=SEED)
    ds = ds.batch(batch_size, drop_remainder=True)
    if repeat:
        ds = ds.repeat()
    return ds.prefetch(tf.data.AUTOTUNE)

print(f"Preprocessing config:")
print(f"  Resolution    : {IMG_HEIGHT}x{IMG_WIDTH}x{CHANNELS}")
print(f"  Normalization : {PIXEL_NORM} -> {'[-1,1]' if PIXEL_NORM=='tanh' else '[0,1]'}")
print(f"  Channel policy: RGBA preserved (alpha channel included in generation target)")
print("Pipeline functions defined. Datasets will be built after split in Section 7.")

---
## Section 6: Dataset Previews

In [ ]:
def show_sprite_grid(hf_ds, n=32, title="Sprites", save_path=None):
    cols = 8; rows = max(1, n // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols*1.6, rows*1.6))
    axes = axes.flatten()
    indices = random.sample(range(len(hf_ds)), min(n, len(hf_ds)))
    for ax, idx in zip(axes, indices):
        arr = np.array(hf_ds[idx]["image"].convert("RGBA"), dtype=np.float32) / 255.0
        ax.imshow(arr); ax.axis("off")
    for ax in axes[len(indices):]: ax.axis("off")
    plt.suptitle(title, fontsize=12); plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.show(); plt.close()

show_sprite_grid(unique_dataset, n=32, title="Original Sprites (128x128 RGBA)",
                 save_path=f"{SAMPLE_DIR}/original_preview.png")
print("Preview saved.")

---
## Section 7: Train / Validation / Test Split

The split convention matches Vanilla GAN: 90% train, 5% validation, 5% test, seed 42. Both notebooks deduplicate in source order and shuffle the resulting unique-image indices deterministically. For exact identity, provide the Vanilla GAN `shuffled_indices.json` through `SHARED_SPLIT_MANIFEST` if available; otherwise the DCGAN notebook reproduces the same split only if dataset ordering and deduplication are identical. The notebook saves its manifest and metadata for later comparisons.

In [ ]:
# Deterministic split compatible with the Vanilla GAN notebook.
# Both notebooks deduplicate in original dataset order, then shuffle the unique-index list with SEED.
# If a shared split manifest is explicitly mounted, use it; otherwise reproduce the same split locally.
SHARED_SPLIT_MANIFEST = ""  # Optional path to Vanilla GAN's shuffled_indices.json
if SHARED_SPLIT_MANIFEST and os.path.isfile(SHARED_SPLIT_MANIFEST):
    print(f"Loading shared split manifest: {SHARED_SPLIT_MANIFEST}")
    with open(SHARED_SPLIT_MANIFEST) as f:
        shuffled = json.load(f)
    if len(shuffled) != len(unique_dataset):
        raise ValueError("Shared split manifest length does not match this deduplicated dataset.")
else:
    if os.path.exists(SHUFFLED_IDX_PATH) and os.path.exists(SPLIT_META_PATH):
        print("Loading cached deterministic split ...")
        with open(SHUFFLED_IDX_PATH) as f:
            shuffled = json.load(f)
    else:
        print(f"Creating deterministic split (seed={SEED}) ...")
        shuffled = list(range(len(unique_dataset)))
        random.Random(SEED).shuffle(shuffled)
        with open(SHUFFLED_IDX_PATH, "w") as f:
            json.dump(shuffled, f)

TOTAL = len(shuffled)
TRAIN_SIZE = int(round(TOTAL * TRAIN_RATIO))
VAL_SIZE = int(round(TOTAL * VAL_RATIO))
TEST_SIZE = TOTAL - TRAIN_SIZE - VAL_SIZE
split_meta = dict(total=TOTAL, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE,
                  seed=SEED, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO,
                  img_height=IMG_HEIGHT, img_width=IMG_WIDTH, channels=CHANNELS,
                  pixel_norm=PIXEL_NORM, manifest_source=(SHARED_SPLIT_MANIFEST or SHUFFLED_IDX_PATH),
                  created_at=datetime.utcnow().isoformat())
with open(SHUFFLED_IDX_PATH, "w") as f:
    json.dump(shuffled, f)
with open(SPLIT_META_PATH, "w") as f:
    json.dump(split_meta, f, indent=2)

train_ids = shuffled[:TRAIN_SIZE]
val_ids = shuffled[TRAIN_SIZE:TRAIN_SIZE + VAL_SIZE]
test_ids = shuffled[TRAIN_SIZE + VAL_SIZE:]
assert not (set(train_ids) & set(val_ids))
assert not (set(train_ids) & set(test_ids))
assert not (set(val_ids) & set(test_ids))

hf_train = unique_dataset.select(train_ids)
hf_val = unique_dataset.select(val_ids)
hf_test = unique_dataset.select(test_ids)

# Build ordinary batched datasets with GLOBAL batch size, then distribute exactly once.
train_ds_unsharded = build_tf_dataset(hf_train, GLOBAL_BATCH_SIZE, shuffle=True, repeat=True)
val_ds_unsharded = build_tf_dataset(hf_val, GLOBAL_BATCH_SIZE, shuffle=False, repeat=False)
train_ds = strategy.experimental_distribute_dataset(train_ds_unsharded)
val_ds = strategy.experimental_distribute_dataset(val_ds_unsharded)
STEPS_PER_EPOCH = TRAIN_SIZE // GLOBAL_BATCH_SIZE
if STEPS_PER_EPOCH < 1:
    raise ValueError("Training split is smaller than one global batch.")

print("Split verified: no overlap")
print(f"Train: {TRAIN_SIZE:,} | Val: {VAL_SIZE:,} | Test: {TEST_SIZE:,}")
print(f"Global batch: {GLOBAL_BATCH_SIZE} | per replica: {BATCH_SIZE_PER_REPLICA} | replicas: {NUM_REPLICAS}")
print(f"Steps/epoch: {STEPS_PER_EPOCH}")
print("Note: identical split indices are reproducible when dataset ordering and deduplication match Vanilla GAN.")

---
## Section 8: DCGAN Architecture

In [ ]:
def dcgan_weight_init(model):
    for layer in model.layers:
        if hasattr(layer,"kernel"):
            layer.kernel.assign(tf.random.normal(layer.kernel.shape,0.0,0.02))
        if hasattr(layer,"bias") and layer.bias is not None:
            layer.bias.assign(tf.zeros_like(layer.bias))
        if isinstance(layer, layers.BatchNormalization):
            if layer.gamma is not None:
                layer.gamma.assign(tf.random.normal(layer.gamma.shape,1.0,0.02))
            if layer.beta is not None:
                layer.beta.assign(tf.zeros_like(layer.beta))

def build_generator(latent_dim,g_channels,init_spatial,img_h,img_w,channels,pixel_norm):
    inp = keras.Input(shape=(latent_dim,),name="z")
    x = layers.Dense(init_spatial*init_spatial*g_channels[0],use_bias=False)(inp)
    x = layers.Reshape((init_spatial,init_spatial,g_channels[0]))(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    for ch in g_channels[1:]:
        x = layers.Conv2DTranspose(ch,4,strides=2,padding="same",use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    act = "tanh" if pixel_norm=="tanh" else "sigmoid"
    x = layers.Conv2DTranspose(channels,4,strides=2,padding="same",activation=act,use_bias=False,name="out")(x)
    return keras.Model(inp,x,name="generator")

def build_discriminator(img_h,img_w,channels,d_channels):
    inp = keras.Input(shape=(img_h,img_w,channels),name="image")
    x = layers.Conv2D(d_channels[0],4,strides=2,padding="same",use_bias=False)(inp)
    x = layers.LeakyReLU(0.2)(x)
    for ch in d_channels[1:]:
        x = layers.Conv2D(ch,4,strides=2,padding="same",use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.LeakyReLU(0.2)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(1,name="logit")(x)
    return keras.Model(inp,x,name="discriminator")

with strategy.scope():
    generator     = build_generator(LATENT_DIM,G_CHANNELS,G_INIT_SPATIAL,IMG_HEIGHT,IMG_WIDTH,CHANNELS,PIXEL_NORM)
    discriminator = build_discriminator(IMG_HEIGHT,IMG_WIDTH,CHANNELS,D_CHANNELS)
    g_optimizer   = keras.optimizers.Adam(G_LR,beta_1=ADAM_BETA1,beta_2=ADAM_BETA2)
    d_optimizer   = keras.optimizers.Adam(D_LR,beta_1=ADAM_BETA1,beta_2=ADAM_BETA2)

if USE_WEIGHT_INIT:
    dcgan_weight_init(generator); dcgan_weight_init(discriminator)
    print("DCGAN weight init applied.")

generator.summary(); discriminator.summary()
print(f"G params: {generator.count_params():,}  D params: {discriminator.count_params():,}")
_z=tf.random.normal((2,LATENT_DIM))
_img=generator(_z,training=False); _log=discriminator(_img,training=False)
assert _img.shape==(2,IMG_HEIGHT,IMG_WIDTH,CHANNELS)
assert _log.shape==(2,1)
print("Forward-pass smoke test: PASSED")

---
## Section 9: Losses and Training Step

In [ ]:
# DCGAN losses and distributed training step.
# Per-example losses are scaled by GLOBAL_BATCH_SIZE so replica gradients combine as a global mean.
_bce_none = keras.losses.BinaryCrossentropy(from_logits=True, reduction=keras.losses.Reduction.NONE)

def _per_example_bce(labels, logits):
    vals = _bce_none(labels, logits)
    return tf.reshape(vals, [-1])

def d_loss_fn(real_logits, fake_logits):
    real_per = _per_example_bce(tf.ones_like(real_logits), real_logits)
    fake_per = _per_example_bce(tf.zeros_like(fake_logits), fake_logits)
    return (tf.nn.compute_average_loss(real_per, global_batch_size=GLOBAL_BATCH_SIZE) +
            tf.nn.compute_average_loss(fake_per, global_batch_size=GLOBAL_BATCH_SIZE))

def g_loss_fn(fake_logits):
    fake_per = _per_example_bce(tf.ones_like(fake_logits), fake_logits)
    return tf.nn.compute_average_loss(fake_per, global_batch_size=GLOBAL_BATCH_SIZE)

@tf.function
def train_step_per_replica(real_images):
    local_batch = tf.shape(real_images)[0]
    z = tf.random.normal((local_batch, LATENT_DIM))
    with tf.GradientTape() as d_tape:
        fake_images = generator(z, training=True)
        real_logits = discriminator(real_images, training=True)
        fake_logits = discriminator(fake_images, training=True)
        d_loss = d_loss_fn(real_logits, fake_logits)
    d_grads = d_tape.gradient(d_loss, discriminator.trainable_variables)
    d_pairs = [(g, v) for g, v in zip(d_grads, discriminator.trainable_variables) if g is not None]
    d_optimizer.apply_gradients(d_pairs)

    with tf.GradientTape() as g_tape:
        fake_images_g = generator(z, training=True)
        fake_logits_g = discriminator(fake_images_g, training=True)
        g_loss = g_loss_fn(fake_logits_g)
    g_grads = g_tape.gradient(g_loss, generator.trainable_variables)
    g_pairs = [(g, v) for g, v in zip(g_grads, generator.trainable_variables) if g is not None]
    g_optimizer.apply_gradients(g_pairs)
    return g_loss, d_loss

@tf.function
def distributed_train_step(distributed_real_images):
    per_g, per_d = strategy.run(train_step_per_replica, args=(distributed_real_images,))
    # SUM combines per-replica losses already scaled by the global batch size.
    g_loss = strategy.reduce(tf.distribute.ReduceOp.SUM, per_g, axis=None)
    d_loss = strategy.reduce(tf.distribute.ReduceOp.SUM, per_d, axis=None)
    return g_loss, d_loss

print("DCGAN losses and distributed training step defined.")
print(f"Distribution strategy: {type(strategy).__name__}, replicas={NUM_REPLICAS}")

---
## Section 10: Checkpoint and Resume Infrastructure

In [ ]:
# Full-state checkpointing and resume infrastructure.
# Optional recovery archive extraction occurs only if this run has no local checkpoint yet.
if RESUME_ARCHIVE_PATH:
    if not os.path.isfile(RESUME_ARCHIVE_PATH):
        raise FileNotFoundError(f"RESUME_ARCHIVE_PATH does not exist: {RESUME_ARCHIVE_PATH}")
    existing_ckpt = tf.train.latest_checkpoint(CKPT_DIR)
    if existing_ckpt:
        print(f"Local checkpoint already exists; not extracting archive over it: {existing_ckpt}")
    else:
        print(f"Restoring artifact archive from {RESUME_ARCHIVE_PATH}")
        with zipfile.ZipFile(RESUME_ARCHIVE_PATH, "r") as zf:
            bad = zf.testzip()
            if bad is not None:
                raise RuntimeError(f"Resume archive is corrupt at member: {bad}")
            # Archive entries are relative to the run root. Reject path traversal.
            root_abs = os.path.abspath(BASE_DIR)
            for member in zf.namelist():
                target = os.path.abspath(os.path.join(BASE_DIR, member))
                if target != root_abs and not target.startswith(root_abs + os.sep):
                    raise RuntimeError(f"Unsafe path in resume archive: {member}")
            zf.extractall(BASE_DIR)
        print("Resume archive extracted. Checkpoint/config compatibility will be checked below.")

ckpt = tf.train.Checkpoint(generator=generator, discriminator=discriminator,
                           g_optimizer=g_optimizer, d_optimizer=d_optimizer)
ckpt_manager = tf.train.CheckpointManager(ckpt, directory=CKPT_DIR,
                                         max_to_keep=3, checkpoint_name="ckpt")
STATE_PATH = f"{CONFIG_DIR}/training_state.json"
HISTORY_CSV = f"{LOG_DIR}/training_history.csv"
HISTORY_JSON = f"{LOG_DIR}/training_history.json"

# Read the restored run configuration BEFORE writing the current target/config.
# This lets us validate architecture/data compatibility while still allowing TARGET_EPOCHS to increase.
RUN_CONFIG_PATH = f"{CONFIG_DIR}/run_config.json"
prior_cfg = None
if os.path.isfile(RUN_CONFIG_PATH):
    try:
        with open(RUN_CONFIG_PATH) as f:
            prior_cfg = json.load(f)
    except Exception as exc:
        raise RuntimeError(f"Could not read saved run configuration: {exc}")

# Save the current effective configuration atomically, preserving resume provenance.
current_cfg = dict(CFG)
current_cfg["resumed_from_config"] = prior_cfg if start_epoch > 0 else None
_cfg_tmp = RUN_CONFIG_PATH + ".tmp"
with open(_cfg_tmp, "w") as f:
    json.dump(current_cfg, f, indent=2)
os.replace(_cfg_tmp, RUN_CONFIG_PATH)

def save_state(epoch, global_step, history, best_epoch, best_g_loss):
    state = dict(epoch=int(epoch), global_step=int(global_step), best_epoch=int(best_epoch),
                 best_g_loss=float(best_g_loss), target_epochs=int(TARGET_EPOCHS),
                 model="DCGAN", img_height=IMG_HEIGHT, img_width=IMG_WIDTH, channels=CHANNELS,
                 latent_dim=LATENT_DIM, g_channels=G_CHANNELS, d_channels=D_CHANNELS,
                 seed=SEED, global_batch_size=GLOBAL_BATCH_SIZE,
                 saved_at=datetime.utcnow().isoformat())
    tmp = STATE_PATH + ".tmp"
    with open(tmp, "w") as f:
        json.dump(state, f, indent=2)
    os.replace(tmp, STATE_PATH)
    tmp_hist = HISTORY_JSON + ".tmp"
    with open(tmp_hist, "w") as f:
        json.dump(history, f, indent=2)
    os.replace(tmp_hist, HISTORY_JSON)

def load_state():
    if os.path.isfile(STATE_PATH):
        with open(STATE_PATH) as f:
            return json.load(f)
    return None

def load_history():
    if os.path.isfile(HISTORY_JSON):
        with open(HISTORY_JSON) as f:
            h = json.load(f)
        return h
    return {"epoch": [], "g_loss": [], "d_loss": [], "epoch_time": [], "global_step": []}

start_epoch = 0
global_step = 0
best_g_loss = float("inf")
best_epoch = -1
history = load_history()
latest = ckpt_manager.latest_checkpoint

if RESUME_MODE == "fresh":
    print("[RESUME] fresh requested. Existing files are preserved, but this run starts at epoch 0.")
elif RESUME_MODE == "required":
    if not latest:
        raise FileNotFoundError(f"RESUME_MODE='required' but no checkpoint found in {CKPT_DIR}")
    status = ckpt.restore(latest)
    status.expect_partial()  # optimizer slot variables can be restored lazily by TF
    state = load_state()
    if not state:
        raise RuntimeError("Checkpoint exists but training_state.json is missing; refusing unsafe resume.")
    if state.get("model") != "DCGAN" or state.get("latent_dim") != LATENT_DIM or state.get("channels") != CHANNELS:
        raise ValueError("Checkpoint architecture/configuration is incompatible with this notebook.")
    start_epoch = int(state["epoch"]) + 1
    global_step = int(state["global_step"])
    best_g_loss = float(state["best_g_loss"])
    best_epoch = int(state["best_epoch"])
    print(f"[RESUME] required: restored after epoch {start_epoch}; global step {global_step}")
elif latest:
    status = ckpt.restore(latest)
    status.expect_partial()
    state = load_state()
    if state:
        if state.get("model") != "DCGAN" or state.get("latent_dim") != LATENT_DIM or state.get("channels") != CHANNELS:
            raise ValueError("Latest checkpoint is incompatible with current DCGAN configuration.")
        start_epoch = int(state["epoch"]) + 1
        global_step = int(state["global_step"])
        best_g_loss = float(state["best_g_loss"])
        best_epoch = int(state["best_epoch"])
        print(f"[RESUME] auto: restored after epoch {start_epoch}; global step {global_step}")
    else:
        raise RuntimeError("Checkpoint exists without training_state.json; refusing to restart silently.")
else:
    print("[RESUME] auto: no checkpoint found; starting at epoch 0")

# Do not silently train on a different architecture/data setup when resuming.
if start_epoch > 0:
    saved_cfg = prior_cfg or {}
    for key, current in [("img_height", IMG_HEIGHT), ("img_width", IMG_WIDTH),
                         ("channels", CHANNELS), ("latent_dim", LATENT_DIM),
                         ("g_channels", G_CHANNELS), ("d_channels", D_CHANNELS),
                         ("global_batch_size", GLOBAL_BATCH_SIZE), ("pixel_norm", PIXEL_NORM),
                         ("hf_dataset", HF_DATASET_NAME)]:
        if key in saved_cfg and saved_cfg[key] != current:
            raise ValueError(f"Resume mismatch for {key}: saved={saved_cfg[key]}, current={current}")

print(f"Will train epochs {start_epoch} through {TARGET_EPOCHS - 1} (target total={TARGET_EPOCHS}).")

---
## Section 11: Checkpoint Smoke Test

In [ ]:
print("--- DCGAN checkpoint smoke test ---")
_smoke_prefix = f"{CKPT_DIR}/smoke_test"
_smoke_ckpt = tf.train.Checkpoint(generator=generator, discriminator=discriminator,
                                  g_optimizer=g_optimizer, d_optimizer=d_optimizer,
                                  smoke_step=tf.Variable(7, dtype=tf.int64))
_smoke_var = generator.trainable_variables[0]
_before = _smoke_var.numpy().copy()
_smoke_ckpt.write(_smoke_prefix)
_smoke_var.assign(tf.zeros_like(_smoke_var))
_smoke_ckpt.smoke_step.assign(0)
_smoke_ckpt.read(_smoke_prefix).expect_partial()
assert np.allclose(_before, _smoke_var.numpy()), "Generator weights did not restore"
assert int(_smoke_ckpt.smoke_step.numpy()) == 7, "Checkpoint metadata did not restore"
for _f in Path(CKPT_DIR).glob("smoke_test*"):
    try: _f.unlink()
    except OSError: pass
print("Checkpoint write/read smoke test: PASSED")
print("Note: full fresh-process optimizer-slot restoration must also be validated on Kaggle before relying on recovery.")

---
## Section 12: Fixed Latent Vectors and Initial Sample Grid

In [ ]:
FIXED_Z_PATH = f"{CONFIG_DIR}/fixed_latent_vectors.npy"
if os.path.exists(FIXED_Z_PATH):
    fixed_z = np.load(FIXED_Z_PATH)
    print(f"Loaded fixed_z: {fixed_z.shape}")
else:
    fixed_z = np.random.RandomState(SEED).randn(N_FIXED_SAMPLES,LATENT_DIM).astype(np.float32)
    np.save(FIXED_Z_PATH,fixed_z)
    print(f"Created fixed_z: {fixed_z.shape}")

def save_sample_grid(label,path):
    imgs = denormalize(generator(fixed_z,training=False).numpy())
    n=len(imgs); cols=5; rows=max(1,n//cols)
    fig,axes=plt.subplots(rows,cols,figsize=(cols*1.8,rows*1.8))
    axes=axes.flatten()
    for ax,img in zip(axes,imgs): ax.imshow(np.clip(img,0,1)); ax.axis("off")
    for ax in axes[n:]: ax.axis("off")
    plt.suptitle(f"DCGAN — {label}",fontsize=11); plt.tight_layout()
    plt.savefig(path,dpi=100,bbox_inches="tight"); plt.close()

save_sample_grid("Epoch 0 (init)",f"{SAMPLE_DIR}/samples_epoch_0000.png")
print("Initial sample grid saved.")

---
## Section 13: Training Loop

In [ ]:
if start_epoch >= TARGET_EPOCHS:
    print(f"Nothing to train: {start_epoch} completed epochs >= target {TARGET_EPOCHS}. Increase TARGET_EPOCHS to continue.")
else:
    if not gpus:
        raise RuntimeError("No GPU detected. Enable Kaggle GPU accelerator before full DCGAN training.")
    print(f"Training epochs {start_epoch + 1} to {TARGET_EPOCHS} | steps/epoch={STEPS_PER_EPOCH}")
    train_iter = iter(train_ds)
    for epoch in range(start_epoch, TARGET_EPOCHS):
        t0 = time.time()
        g_losses, d_losses = [], []
        try:
            for _ in range(STEPS_PER_EPOCH):
                g_l, d_l = distributed_train_step(next(train_iter))
                g_val, d_val = float(g_l.numpy()), float(d_l.numpy())
                if not (np.isfinite(g_val) and np.isfinite(d_val)):
                    raise FloatingPointError(f"Non-finite loss at epoch {epoch + 1}, step {global_step}")
                g_losses.append(g_val); d_losses.append(d_val)
                global_step += 1
        except KeyboardInterrupt:
            print("Interrupted during epoch. The last completed-epoch checkpoint remains valid; current partial epoch will be retried on resume.")
            raise

        mg, md_ = float(np.mean(g_losses)), float(np.mean(d_losses))
        elapsed = time.time() - t0
        history["epoch"].append(epoch)
        history["g_loss"].append(mg)
        history["d_loss"].append(md_)
        history["epoch_time"].append(elapsed)
        history["global_step"].append(global_step)

        if mg < best_g_loss:
            best_g_loss, best_epoch = mg, epoch
            generator.save_weights(f"{WEIGHTS_DIR}/generator_best.weights.h5")
            discriminator.save_weights(f"{WEIGHTS_DIR}/discriminator_best.weights.h5")

        # Rewrite history atomically from the authoritative in-memory state, avoiding duplicate rows on resume.
        hist_df = pd.DataFrame(history)
        hist_df.to_csv(HISTORY_CSV, index=False)
        with open(HISTORY_JSON + ".tmp", "w") as f:
            json.dump(history, f, indent=2)
        os.replace(HISTORY_JSON + ".tmp", HISTORY_JSON)

        if (epoch + 1) % CHECKPOINT_EVERY == 0:
            ckpt_manager.save(checkpoint_number=epoch)
            save_state(epoch, global_step, history, best_epoch, best_g_loss)
        if (epoch + 1) % SAMPLE_EVERY == 0 or epoch == TARGET_EPOCHS - 1:
            save_sample_grid(f"Epoch {epoch + 1}", f"{SAMPLE_DIR}/samples_epoch_{epoch + 1:04d}.png")
        print(f"Epoch {epoch + 1:3d}/{TARGET_EPOCHS} | G:{mg:.4f} D:{md_:.4f} | {elapsed:.1f}s | step={global_step}")

    print(f"Training loop finished. Completed history epochs: {len(history['epoch'])}; best G loss={best_g_loss:.4f} at epoch {best_epoch + 1 if best_epoch >= 0 else 'N/A'}")

---
## Section 14: Monitoring — Loss Curves

In [ ]:
hist_df=pd.read_csv(HISTORY_CSV)
fig,axes=plt.subplots(1,2,figsize=(14,4))
axes[0].plot(hist_df["epoch"]+1,hist_df["g_loss"],label="G Loss")
axes[0].plot(hist_df["epoch"]+1,hist_df["d_loss"],label="D Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("DCGAN Training Losses"); axes[0].legend()
axes[1].plot(hist_df["epoch"]+1,hist_df["epoch_time"])
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Seconds")
axes[1].set_title("Epoch Duration")
plt.tight_layout()
plt.savefig(f"{LOG_DIR}/loss_curves.png",dpi=120,bbox_inches="tight")
plt.show(); plt.close()
print("Loss curves saved.")
if hist_df["g_loss"].iloc[-5:].std()>1.0:
    print("[WARN] G loss highly variable — possible instability.")
if hist_df["d_loss"].iloc[-1]<0.01:
    print("[WARN] D loss near zero — discriminator may have collapsed.")

---
## Section 15: Final Model Selection

In [ ]:
generator.save_weights(f"{WEIGHTS_DIR}/generator_latest.weights.h5")
discriminator.save_weights(f"{WEIGHTS_DIR}/discriminator_latest.weights.h5")
generator.save(f"{WEIGHTS_DIR}/generator_latest.keras")
discriminator.save(f"{WEIGHTS_DIR}/discriminator_latest.keras")
print("Latest weights saved.")
print(f"Best checkpoint: epoch {best_epoch+1}, G loss {best_g_loss:.4f}")
print("NOTE: Inspect sample grids to visually confirm best epoch.")

---
## Section 16: Evaluation

In [ ]:
import hashlib as _hl
rng_eval  = np.random.RandomState(SEED+1)
eval_z    = rng_eval.randn(N_EVAL_SAMPLES,LATENT_DIM).astype(np.float32)
eval_imgs = denormalize(generator(eval_z,training=False).numpy())

flat_eval     = eval_imgs.reshape(N_EVAL_SAMPLES,-1)
diversity_std = float(flat_eval.std(axis=0).mean())
print(f"Diversity (mean pixel std): {diversity_std:.4f}")
if diversity_std<0.01: print("[WARN] Very low diversity — possible mode collapse.")

gen_hashes   = [_hl.sha256((img*255).astype("uint8").tobytes()).hexdigest() for img in eval_imgs]
n_unique_gen = len(set(gen_hashes))
print(f"Unique generated: {n_unique_gen}/{N_EVAL_SAMPLES}")

print("Computing NN distances vs 200 test sprites ...")
test_items = [hf_test[i] for i in range(min(200,len(hf_test)))]
test_arrs  = np.stack([denormalize(preprocess_image(it)) for it in test_items])
test_flat  = test_arrs.reshape(len(test_arrs),-1)
gen_flat   = eval_imgs[:len(test_arrs)].reshape(len(test_arrs),-1)
nn_dists   = np.linalg.norm(gen_flat[:,None,:]-test_flat[None,:,:],axis=-1).min(axis=1)
print(f"Mean NN dist (L2): {nn_dists.mean():.4f}  std: {nn_dists.std():.4f}")

eval_report = dict(model="DCGAN",epochs_completed=len(history["epoch"]),
    best_epoch=best_epoch,best_g_loss=best_g_loss,
    diversity_std=diversity_std,unique_generated=n_unique_gen,total_generated=N_EVAL_SAMPLES,
    nn_mean_dist=float(nn_dists.mean()),nn_std_dist=float(nn_dists.std()),
    latent_dim=LATENT_DIM,img_size=f"{IMG_HEIGHT}x{IMG_WIDTH}x{CHANNELS}",
    g_channels=G_CHANNELS,d_channels=D_CHANNELS)
with open(f"{LOG_DIR}/evaluation_report.json","w") as f: json.dump(eval_report,f,indent=2)
print("Evaluation report saved.")

---
## Section 17: Artifact Export

In [ ]:
def export_all_artifacts():
    """Export current state without fabricating a completed epoch when training has not started."""
    last_completed = history["epoch"][-1] if history["epoch"] else None
    if last_completed is not None:
        ckpt_manager.save(checkpoint_number=int(last_completed))
        save_state(last_completed, global_step, history, best_epoch, best_g_loss)
    generator.save_weights(f"{WEIGHTS_DIR}/generator_latest.weights.h5")
    discriminator.save_weights(f"{WEIGHTS_DIR}/discriminator_latest.weights.h5")
    generator.save(f"{WEIGHTS_DIR}/generator_latest.keras")
    discriminator.save(f"{WEIGHTS_DIR}/discriminator_latest.keras")
    label = f"Epoch {last_completed + 1}" if last_completed is not None else "Initial untrained model"
    save_sample_grid(label, f"{SAMPLE_DIR}/samples_final.png")

    import io as _io
    for model, name in [(generator, "generator"), (discriminator, "discriminator")]:
        buf = _io.StringIO()
        model.summary(print_fn=lambda line: buf.write(line + "\n"))
        with open(f"{CONFIG_DIR}/{name}_summary.txt", "w") as f:
            f.write(buf.getvalue())

    ds_info = dict(source=HF_DATASET_NAME, unique_images=len(unique_dataset),
                   train=TRAIN_SIZE, val=VAL_SIZE, test=TEST_SIZE,
                   img_height=IMG_HEIGHT, img_width=IMG_WIDTH, channels=CHANNELS,
                   pixel_norm=PIXEL_NORM, seed=SEED,
                   train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO,
                   global_batch_size=GLOBAL_BATCH_SIZE, batch_size_per_replica=BATCH_SIZE_PER_REPLICA)
    with open(f"{CONFIG_DIR}/dataset_info.json", "w") as f:
        json.dump(ds_info, f, indent=2)

    # Explicitly preserve the deterministic index manifest and split metadata in the archive.
    for source, target_name in [(SHUFFLED_IDX_PATH, "shuffled_indices.json"),
                                (SPLIT_META_PATH, "split_metadata.json"),
                                (UNIQUE_IDX_PATH, "unique_indices.json")]:
        if os.path.isfile(source):
            target = os.path.join(CONFIG_DIR, target_name)
            if os.path.abspath(source) != os.path.abspath(target):
                shutil.copy2(source, target)

    completed_epochs = (last_completed + 1) if last_completed is not None else 0
    readme = (
        "# DCGAN 280K Artifact Bundle\n\n"
        "This archive contains DCGAN model files, configuration, logs, samples, and resumable TensorFlow checkpoints when training has completed at least one epoch.\n\n"
        "## Resume in a fresh Kaggle session\n"
        "1. Attach this ZIP as a Kaggle input dataset.\n"
        "2. Set RESUME_ARCHIVE_PATH to the mounted ZIP path in Section 3.\n"
        "3. Set TARGET_EPOCHS=100 (total target) and RESUME_MODE='auto' or 'required'.\n"
        "4. Run cells in order. The archive is extracted only when no local checkpoint exists.\n\n"
        f"Completed epochs at export: {completed_epochs}.\n\n"
        "## Important\n"
        "Kaggle /kaggle/working storage is temporary. Download the ZIP or save a versioned Kaggle output before the session ends.\n"
    )
    with open(f"{BASE_DIR}/README.md", "w") as f:
        f.write(readme)
    print(f"Artifact files exported. Completed epochs: {completed_epochs}.")

export_all_artifacts()

---
## Section 18: ZIP Creation and Verification

In [ ]:
import hashlib as _hl

REQUIRED_PATTERNS = [
    "weights/generator_latest.keras",
    "weights/generator_latest.weights.h5",
    "weights/discriminator_latest.keras",
    "weights/discriminator_latest.weights.h5",
    "checkpoints/checkpoint",
    "logs/training_history.csv",
    "logs/training_history.json",
    "config/run_config.json",
    "config/fixed_latent_vectors.npy",
    "config/training_state.json",
    "config/shuffled_indices.json",
    "config/split_metadata.json",
    "samples/samples_final.png",
    "README.md",
]

def _sha256_file(path):
    h = _hl.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def create_and_verify_zip(zip_path, source_dir):
    # Inventory excludes the ZIP itself and its staging file if either lives nearby.
    inv = []
    for root, _, files in os.walk(source_dir):
        for fname in files:
            full = os.path.join(root, fname)
            rel = os.path.relpath(full, source_dir)
            inv.append({"path": rel, "size_bytes": os.path.getsize(full), "sha256": _sha256_file(full)})
    manifest_path = os.path.join(source_dir, "artifact_manifest.json")
    with open(manifest_path + ".tmp", "w") as f:
        json.dump(inv, f, indent=2)
    os.replace(manifest_path + ".tmp", manifest_path)

    staging = zip_path + ".staging"
    try:
        with zipfile.ZipFile(staging, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=4) as zf:
            for root, _, files in os.walk(source_dir):
                for fname in files:
                    full = os.path.join(root, fname)
                    zf.write(full, os.path.relpath(full, source_dir))
        with zipfile.ZipFile(staging, "r") as zf:
            bad = zf.testzip()
            if bad is not None:
                raise RuntimeError(f"ZIP integrity failure at member: {bad}")
            names = set(zf.namelist())
            missing = [p for p in REQUIRED_PATTERNS if not any(n.endswith(p) for n in names)]
            if missing:
                raise RuntimeError(f"ZIP missing required artifacts: {missing}")
        # Only replace the previous archive after the staged archive passes checks.
        os.replace(staging, zip_path)
        size_mb = os.path.getsize(zip_path) / 1e6
        print(f"ZIP verified: {zip_path}")
        print(f"Size: {size_mb:.2f} MB | files: {len(names)} | required artifacts: PASS | CRC test: PASS")
        return zip_path, size_mb
    except Exception:
        if os.path.exists(staging):
            os.remove(staging)
        raise

zip_path, zip_mb = create_and_verify_zip(ZIP_PATH, BASE_DIR)

---
## Section 19: Download and Persistence

In [ ]:
from IPython.display import display, FileLink, Markdown

if not os.path.isfile(ZIP_PATH):
    raise FileNotFoundError(f"Expected archive not found: {ZIP_PATH}")
print(f"Archive created and integrity-checked: {ZIP_PATH}")
print(f"Archive size: {os.path.getsize(ZIP_PATH) / 1e6:.2f} MB")
print("Download status: the file is available in this notebook session; it is not confirmed downloaded until you click the link.")
display(FileLink(ZIP_PATH, result_html_prefix="Download DCGAN archive: "))
print("For persistence: save/commit a Kaggle notebook version with output files, or download this ZIP before the session ends. /kaggle/working is temporary.")

---
## Section 20: Resume Instructions

In [ ]:
st=load_state()
if st:
    print(f"Last epoch : {st['epoch']+1}")
    print(f"Global step: {st['global_step']}")
    print(f"Best G loss: {st['best_g_loss']:.4f} (epoch {st['best_epoch']+1})")
else:
    print("No state found yet.")

print("""
--- Resume from epoch 50 to 100 ---
1. Download DCGAN_280K_COMPLETE.zip from Output tab.
2. Upload as Kaggle dataset (My Datasets -> New Dataset).
3. In new session, copy files:
     import shutil, glob
     for f in glob.glob("/kaggle/input/<name>/checkpoints/*"):
         shutil.copy(f, "/kaggle/working/dcgan_280k/checkpoints/")
     for f in glob.glob("/kaggle/input/<name>/config/*"):
         shutil.copy(f, "/kaggle/working/dcgan_280k/config/")
4. Set TARGET_EPOCHS=100, RESUME_MODE="auto" in Section 3.
5. Run all cells.
NOTE: TARGET_EPOCHS is TOTAL target, not additional epochs.
""")

---
## Section 21: Final Experiment Report

In [ ]:
hist_df=pd.read_csv(HISTORY_CSV) if os.path.exists(HISTORY_CSV) else pd.DataFrame()

report=dict(
    experiment="DCGAN 280K Alucard Sprites",
    dataset=HF_DATASET_NAME, unique_images=len(unique_dataset),
    train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE,
    img_resolution=f"{IMG_HEIGHT}x{IMG_WIDTH}x{CHANNELS}",
    pixel_norm=PIXEL_NORM, latent_dim=LATENT_DIM,
    g_channels=G_CHANNELS, d_channels=D_CHANNELS, g_init_spatial=G_INIT_SPATIAL,
    g_params=int(generator.count_params()), d_params=int(discriminator.count_params()),
    g_lr=G_LR, d_lr=D_LR, beta1=ADAM_BETA1, beta2=ADAM_BETA2,
    global_batch_size=GLOBAL_BATCH_SIZE,
    epochs_completed=len(hist_df) if len(hist_df) else 0,
    target_epochs=TARGET_EPOCHS, best_epoch=best_epoch, best_g_loss=best_g_loss,
    final_g_loss=float(hist_df["g_loss"].iloc[-1]) if len(hist_df) else None,
    final_d_loss=float(hist_df["d_loss"].iloc[-1]) if len(hist_df) else None,
    zip_path=ZIP_PATH,
    zip_size_mb=round(os.path.getsize(ZIP_PATH)/1e6,1) if os.path.exists(ZIP_PATH) else None,
    tf_version=tf.__version__, num_gpus=len(gpus),
    generated_at=datetime.utcnow().isoformat()
)
with open(f"{LOG_DIR}/final_report.json","w") as f: json.dump(report,f,indent=2)

print("="*60)
print("DCGAN 280K — FINAL EXPERIMENT REPORT")
print("="*60)
for k,v in report.items(): print(f"  {k:<28}: {v}")
print("="*60)
print(f"Artifact ZIP: {ZIP_PATH}")
print("All done. Download the ZIP from the Kaggle Output tab.")